# Preprocessing & Dataset Preparation

This notebook prepares the dataset for a separate training notebook.

## Responsibilities of this notebook
- Load and clean the raw dataset.
- Perform **deterministic, row-level feature extraction/aggregation** that does not learn from the dataset.
- Split the data into stratified training and test sets.
- Define the reusable preprocessing configuration for numerical and categorical columns.
- Save the prepared train/test datasets and preprocessing metadata.

## 1. Imports and configuration

In [114]:
# ------------------------- imports -------------------------
import json, os, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTENC
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [115]:
# ------------------------- Paths -------------------------
DATA = '../Data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
ARTIFACTS_DIR = '../Artifacts/'
DATAFRAMES_DIR = '../DataFrames'

for d in ([ARTIFACTS_DIR, DATAFRAMES_DIR]):
    os.makedirs(d, exist_ok=True)
print("All project directories confirmed and ready.")

All project directories confirmed and ready.


In [116]:
# ------------------------- Var -------------------------
TARGET = "Churn"
RANDOM_STATE = 42
TEST_SIZE = 0.20

## 2. Load and clean the raw data

In [117]:
df = pd.read_csv(DATA).drop(columns=["customerID"])

# TotalCharges is delivered as text in the raw dataset; convert invalid/blank values to NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Shape of the DataFrame:", df.shape)
display(df.sample(5, random_state=RANDOM_STATE))


Shape of the DataFrame: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,Female,0,Yes,No,1,No,No phone service,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.80,Yes
2715,Male,0,No,No,41,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,Female,0,Yes,Yes,52,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.70,No
1807,Female,0,No,No,1,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,Male,0,No,No,67,Yes,No,DSL,No,No,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.10,No


In [118]:
# Remove exact duplicate rows.
print("Duplicate rows before removal:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after duplicate removal:", df.shape)


Duplicate rows before removal: 22
Shape after duplicate removal: (7021, 20)


## 3. Handle the Missing Values in `TotalCharges`

In EDA, we determined that `TotalCharges` ≈ `MonthlyCharges` * `tenure` for all customers, and specifically for Customers with `TotalChanges` *= N/A*

Based on that, `0` is the correct value to fill the missing values.

In [119]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
missing_percent = df.isnull().mean() * 100
missing = df.isnull().sum()

missing_table = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percent
})

print(missing_table.sort_values("Missing %", ascending=False))

                  Missing Count  Missing %
gender                        0        0.0
SeniorCitizen                 0        0.0
Partner                       0        0.0
Dependents                    0        0.0
tenure                        0        0.0
PhoneService                  0        0.0
MultipleLines                 0        0.0
InternetService               0        0.0
OnlineSecurity                0        0.0
OnlineBackup                  0        0.0
DeviceProtection              0        0.0
TechSupport                   0        0.0
StreamingTV                   0        0.0
StreamingMovies               0        0.0
Contract                      0        0.0
PaperlessBilling              0        0.0
PaymentMethod                 0        0.0
MonthlyCharges                0        0.0
TotalCharges                  0        0.0
Churn                         0        0.0


## 4. Deterministic feature extraction & aggregation

These features depend only on the values of the current customer. They do **not** use dataset-wide statistics, so they can safely be created before the train/test split.

In [120]:
SERVICE_COLS = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

# Service usage count: counts active services only.
df["ServiceCount"] = sum(
    (df[col] == "Yes").astype(int)
    for col in SERVICE_COLS
)

In [121]:
# Entertainment/streaming count.
df["StreamingCount"] = (
    (df["StreamingTV"] == "Yes").astype(int)
    + (df["StreamingMovies"] == "Yes").astype(int)
)

In [122]:
# Security/support count.
df["SecuritySupportCount"] = (
    (df["OnlineSecurity"] == "Yes").astype(int)
    + (df["TechSupport"] == "Yes").astype(int)
    + (df["DeviceProtection"] == "Yes").astype(int)
)

In [123]:
# Household/family indicator.
df["HasFamily"] = (
    (df["Partner"] == "Yes") | (df["Dependents"] == "Yes")
).astype(int)

In [124]:
# Fixed business-rule tenure flag: no dataset statistic is learned.
df["IsNewCustomer"] = (df["tenure"] <= 6).astype(int)

In [125]:
# Derived spending relationship. For tenure=0, use the current monthly charge.
# df["AverageMonthlySpend"] = np.where(
#     df["tenure"] > 0,
#     df["TotalCharges"] / df["tenure"],
#     df["MonthlyCharges"]
# )

In [126]:
engineered_cols = [
    "ServiceCount",
    "StreamingCount",
    "SecuritySupportCount",
    "HasFamily",
    "IsNewCustomer",
    # "AverageMonthlySpend",
]
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(include="str").columns

print(f'Dataset *NEW* Shape -> Customers: {df.shape[0]}, Features (without Target): {df.shape[1]}')

print("\nNumerical columns:")
print(numeric_cols.tolist())

print("\nCategorical columns (with Traget):")
print(categorical_cols.tolist())

print()

display(df[engineered_cols].head())

Dataset *NEW* Shape -> Customers: 7021, Features (without Target): 25

Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'ServiceCount', 'StreamingCount', 'SecuritySupportCount', 'HasFamily', 'IsNewCustomer']

Categorical columns (with Traget):
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']



,ServiceCount,StreamingCount,SecuritySupportCount,HasFamily,IsNewCustomer
0,1,0,0,1,1
1,3,0,2,0,0
2,3,0,1,0,1
3,3,0,3,0,0
4,1,0,0,0,1


## 5. Separate features (`X`) and target (`y`), Create Train/test Split

In [127]:
X = df.drop(columns=TARGET).copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target distribution:")
display(y.value_counts())
display(y.value_counts(normalize=True).rename("proportion"))


X shape: (7021, 24)
y shape: (7021,)
Target distribution:


Churn
No     5164
Yes    1857
Name: count, dtype: int64

Churn
No     0.735508
Yes    0.264492
Name: proportion, dtype: float64

In [128]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

Train: X=(5616, 24), y=(5616,)
Test:  X=(1405, 24), y=(1405,)


In [129]:
split_summary = pd.DataFrame({
    "Train": y_train.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True),
}).rename_axis("Churn")

display(split_summary)

,Train,Test
Churn,,
No,0.735577,0.735231
Yes,0.264423,0.264769


## 6. Features that must be learned from training data

In [130]:
def group_values(feature_name, bins_num, X_train, df_save):
    min_value = X_train[feature_name].min()
    max_value = X_train[feature_name].max()

    bins = np.linspace(
        min_value,
        max_value,
        bins_num + 1
    )

    labels = [
        f"{bins[i]:.0f}-{bins[i+1]:.0f}"
        for i in range(len(bins) - 1)
    ]

    print("Labels for the ranges:", labels)

    name = f"{feature_name}Group"

    df_save[name] = pd.cut(
        df_save[feature_name],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return name

In [131]:
# Spending
median_mon_charges = X_train["MonthlyCharges"].median()

X_train["NewCustomer_HighCharge"] = ((X_train["tenure"] <= 6) &
 (X_train["MonthlyCharges"] > median_mon_charges)).astype(int)

X_test["NewCustomer_HighCharge"] = ((X_test["tenure"] <= 6) &
 (X_test["MonthlyCharges"] > median_mon_charges)).astype(int)

In [132]:
# Group MonthlyCharge values
# group_monthlycharge_train = group_values('MonthlyCharges', 5,X_train, X_train)
# group_totalcharge_train = group_values('TotalCharges',5, X_train, X_train)

# # Group TotalCharge values
# group_monthlycharge_test = group_values('MonthlyCharges', 5, X_train, X_test)
# group_totalcharge_train = group_values('TotalCharges',5, X_train, X_test)

In [133]:
# Tenure
# median_tenure = X_train["tenure"].median()

# X_train["IsOldCustomer"] = (X_train["tenure"] > median_tenure).astype(int)
# X_test["IsOldCustomer"] = (X_test["tenure"] > median_tenure).astype(int)

# Group TotalCharge values
# group_tenure_train = group_values('tenure',5,X_train, X_train)
# group_tenure_test = group_values('tenure',5,X_train, X_test)

In [134]:
engineered_cols = [
    'NewCustomer_HighCharge',
    # group_monthlycharge_train,
    # group_totalcharge_train,
    # group_tenure_train,
]

numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(include="str").columns

print(f'Dataset *NEWEST* Shape -> Customers: {df.shape[0]}, Features (without Target): {df.shape[1]}')

print("\nNumerical columns:")
print(numeric_cols.tolist())

print("\nCategorical columns (with Traget):")
print(categorical_cols.tolist())

print()

display(X_train[engineered_cols].head())

Dataset *NEWEST* Shape -> Customers: 7021, Features (without Target): 25

Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'ServiceCount', 'StreamingCount', 'SecuritySupportCount', 'HasFamily', 'IsNewCustomer']

Categorical columns (with Traget):
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']



,NewCustomer_HighCharge
2624,1
2645,0
1416,0
5760,0
2420,0


## 7. Correlation Test

In [135]:
# Using df.corr()
numeric_cols = ['tenure', 'SecuritySupportCount', 'StreamingCount',
 'TotalCharges', 'MonthlyCharges']
X_train[numeric_cols].corr()

,tenure,SecuritySupportCount,StreamingCount,TotalCharges,MonthlyCharges
tenure,1.000000,0.466422,0.325917,0.829330,0.250913
SecuritySupportCount,0.466422,1.000000,0.450791,0.621977,0.508344
StreamingCount,0.325917,0.450791,1.000000,0.592378,0.716706
TotalCharges,0.829330,0.621977,0.592378,1.000000,0.651351
MonthlyCharges,0.250913,0.508344,0.716706,0.651351,1.000000


In [136]:
# Using VIF
X_num_const = sm.add_constant(X_train[numeric_cols])
vif = pd.DataFrame({
    'feature': X_num_const.columns,
    'VIF': [variance_inflation_factor(X_num_const.values, i) for i in range(X_num_const.shape[1])]
})

vif = vif[vif['feature'] != 'const']
vif.sort_values('VIF', ascending=False)

,feature,VIF
4,TotalCharges,10.235470
1,tenure,5.993063
5,MonthlyCharges,4.017775
3,StreamingCount,2.183126
2,SecuritySupportCount,1.685064


*The Rest of the Tesing is in the Next Section*

## 8. Define the reusable preprocessing configuration

This section **defines** the preprocessing steps but does not fit them.

Processing design:
- Continuous numeric features → median imputation → Yeo-Johnson → standardization.
- Binary numeric features are treated as categorical/binary and are not skew-corrected.
- Categorical features → most-frequent imputation → one-hot encoding.
- `drop="if_binary"` reduces binary categorical variables to one encoded column.
- `handle_unknown="ignore"` keeps inference safe when a new category appears.

In [137]:
numeric_cols = ['tenure', 'SecuritySupportCount', 'StreamingCount',
 'TotalCharges', 'MonthlyCharges']

categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 
 'PaymentMethod','HasFamily', 'IsNewCustomer', 'NewCustomer_HighCharge']

In [138]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    # ('smote', SMOTENC(random_state=RANDOM_STATE)),
    ("yeo_johnson", PowerTransformer(method="yeo-johnson")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # ('smote', SMOTENC(random_state=RANDOM_STATE)),
    ("encoder", OneHotEncoder(
        drop="if_binary",
        handle_unknown="ignore",
    )),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    # ("binary_numeric", "passthrough", binary_numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

## 9. Feature Importance

In [139]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

model = rf_pipeline.fit(X_train, y_train)

rf = model.named_steps["model"]
importances = rf.feature_importances_

feature_names = model.named_steps["preprocessor"].get_feature_names_out()

feature_importance = (
    pd.Series(importances, index=feature_names)
    .sort_values(ascending=False)
)

feature_importance

numeric__TotalCharges                                   0.149899
numeric__MonthlyCharges                                 0.138946
numeric__tenure                                         0.129962
categorical__Contract_Month-to-month                    0.061366
categorical__OnlineSecurity_No                          0.031462
categorical__PaymentMethod_Electronic check             0.026843
categorical__gender_Male                                0.026646
categorical__InternetService_Fiber optic                0.026144
categorical__PaperlessBilling_Yes                       0.024987
categorical__IsNewCustomer_1                            0.023683
categorical__SeniorCitizen_1                            0.020285
categorical__TechSupport_No                             0.020141
numeric__SecuritySupportCount                           0.019996
categorical__OnlineBackup_No                            0.017454
categorical__Partner_Yes                                0.016352
categorical__Dependents_Y

## 10. Make the KFold Splits

In [140]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print('KFold Splits: ',skf)

KFold Splits:  StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


## 11. Save prepared data and preprocessing template

The preprocessing object saved here is **unfitted**. This notebook does not train and does not fit preprocessing parameters.

In [141]:
# Save split data separately so the target is not accidentally treated as an input feature.
X_train.to_parquet(os.path.join(DATAFRAMES_DIR, "X_train.parquet"))
X_test.to_parquet(os.path.join(DATAFRAMES_DIR, "X_test.parquet"))
y_train.to_frame(TARGET).to_parquet(os.path.join(DATAFRAMES_DIR, "y_train.parquet"))
y_test.to_frame(TARGET).to_parquet(os.path.join(DATAFRAMES_DIR, "y_test.parquet"))

In [142]:
# Save an UNFITTED preprocessing template for the training notebook.
joblib.dump(preprocessor, os.path.join(ARTIFACTS_DIR, "preprocessor_template.pkl"))

# Save the KFold Splits
joblib.dump(skf, os.path.join(ARTIFACTS_DIR, "skf.pkl"))

['../Artifacts/skf.pkl']